# CART 如何选择切分？

**面试回答：**CART 枚举特征阈值，以加权 Gini 下降选择当前节点切分；它是贪心局部最优，因此需用深度和最小叶子样本数控制方差。

## 真实案例

贷款预审用月收入与近三月逾期数判断人工复核，叶子输出历史复核比例。

In [1]:
import numpy as np  # 导入 NumPy 手写 CART 根节点。
app=np.array(['L01','L02','L03','L04','L05','L06','L07','L08','V01','V02'])  # 构造申请编号。
x=np.array([[5,0],[7,0],[8,1],[10,0],[12,2],[15,1],[18,3],[20,0],[9,2],[17,0]],dtype=float)  # 记录收入万元和逾期数。
y=np.array([0,0,1,0,1,1,1,0,1,0])  # 标记是否进入人工复核。
print('申请 | 收入万 | 逾期 | 复核')  # 输出数据表头。
for n,row,c in zip(app,x,y):  # 逐条展示申请。
    print(n,row.tolist(),c)  # 输出一条业务记录。

申请 | 收入万 | 逾期 | 复核
L01 [5.0, 0.0] 0
L02 [7.0, 0.0] 0
L03 [8.0, 1.0] 1
L04 [10.0, 0.0] 0
L05 [12.0, 2.0] 1
L06 [15.0, 1.0] 1
L07 [18.0, 3.0] 1
L08 [20.0, 0.0] 0
V01 [9.0, 2.0] 1
V02 [17.0, 0.0] 0


## Baseline / 基线

基线只看逾期数是否大于零。

In [2]:
train=np.arange(8)  # 选择历史申请。
valid=np.arange(8,10)  # 选择回放申请。
baseline=(x[valid,1]>0).astype(int)  # 用单规则输出复核结果。
baseline_acc=float(np.mean(baseline==y[valid]))  # 计算规则准确率。
print('规则基线:',baseline.tolist(),baseline_acc)  # 输出基线。

规则基线: [1, 0] 1.0


In [3]:
def gini(target):  # 定义二分类 Gini 不纯度。
    p=target.mean() if len(target) else 0.0  # 计算正类比例并处理空集。
    return 1-p*p-(1-p)*(1-p)  # 返回不纯度。
def best_split(data,target):  # 枚举 CART 根节点候选。
    best=(1e9,None,None)  # 初始化最小加权不纯度。
    for feature in range(data.shape[1]):  # 遍历特征列。
        for threshold in np.unique(data[:,feature])[:-1]:  # 遍历可切分阈值。
            left=data[:,feature]<=threshold  # 标识左子节点。
            score=left.mean()*gini(target[left])+(~left).mean()*gini(target[~left])  # 计算加权子节点不纯度。
            if score<best[0]:  # 判断当前切分是否更纯。
                best=(score,feature,threshold)  # 保存最优根节点切分。
    return best  # 返回不纯度、特征和阈值。
score,feature,threshold=best_split(x[train],y[train])  # 在历史申请上选择根节点。
left=x[train,feature]<=threshold  # 计算训练根节点左右掩码。
left_value=int(y[train][left].mean()>=.5)  # 得到左叶多数类。
right_value=int(y[train][~left].mean()>=.5)  # 得到右叶多数类。
pred=np.where(x[valid,feature]<=threshold,left_value,right_value)  # 对回放申请输出根节点树预测。
acc=float(np.mean(pred==y[valid]))  # 计算 CART 回放准确率。
print('根节点特征/阈值/Gini:',feature,threshold,round(score,3))  # 输出切分中间量。
print('左右叶预测:',left_value,right_value,'验证预测:',pred.tolist())  # 输出树规则。

根节点特征/阈值/Gini: 1 0.0 0.0
左右叶预测: 0 1 验证预测: [1, 0]


## 结果解读

Gini 下降选的是当前节点最纯的局部规则，不保证全局最优。叶子样本少时，叶子比例不应直接当作校准风险。

In [4]:
print('模型 | 回放准确率 | 规则')  # 输出比较表头。
print(f'逾期阈值 | {baseline_acc:.2f} | 逾期>0')  # 输出基线行。
print(f'CART根节点 | {acc:.2f} | 特征{feature} <= {threshold}')  # 输出 CART 行。
print('生产差距：应限制深度/最小叶子、做时间验证、审计规则与缺失值路径。')  # 说明生产边界。

模型 | 回放准确率 | 规则
逾期阈值 | 1.00 | 逾期>0
CART根节点 | 1.00 | 特征1 <= 0.0
生产差距：应限制深度/最小叶子、做时间验证、审计规则与缺失值路径。


## 失败案例与修复

若允许单样本叶子，树可按高基数 ID 记忆训练记录；修复是拒绝 ID、设置最小叶子并在独立时间窗口验证。

In [5]:
id_feature=np.arange(len(train),dtype=float)  # 构造每条申请唯一的高基数伪特征。
id_score,id_col,id_threshold=best_split(np.c_[x[train],id_feature],y[train])  # 让树错误地把 ID 当候选特征。
print('失败：加入ID后的候选根特征=',id_col,'阈值=',id_threshold)  # 输出 ID 泄漏风险。
print('修复：特征白名单只保留预测时稳定、可解释的业务字段。')  # 输出修复原则。
print('注意：本例仅一层树，完整 CART 还需要递归和剪枝。')  # 说明教学简化。

失败：加入ID后的候选根特征= 1 阈值= 0.0
修复：特征白名单只保留预测时稳定、可解释的业务字段。
注意：本例仅一层树，完整 CART 还需要递归和剪枝。


In [6]:
assert len(app)>=5  # 保护样本数量。
assert score<=gini(y[train])  # 保护切分不增加根节点不纯度。
assert acc>=baseline_acc  # 保护树不弱于基线。
assert feature in [0,1]  # 保护正式模型未使用伪 ID。